# SingBERT: style classifier (`english` | `singlish`)

Runs [`training/style_singbert.py`](style_singbert.py): fine-tunes `zanelim/singbert` on the user turns
of the synthetic and MultiWOZ `[TRACK]` files (Singlish user lines vs. their English normalized lines
and MultiWOZ turns), evaluates it on dev through the backend's own `detect_style` next to the keyword
stub, and pushes it to `makan-where-cs425/makan-singbert-style-classifier`, which the backend serves as `MAKAN_STYLE_MODEL`.
All training code is in the script; this notebook only sets up and calls it.

- **Colab:** GPU runtime (T4 is plenty: a full run takes a few minutes), secrets `HF_TOKEN` (write access,
  to push) and `GITHUB_TOKEN` (read access, to clone the repo), each with notebook access. Checkpoints go
  to Drive, so after a disconnect, run all again and training resumes from the newest one.
- **GPU cluster / any Jupyter:** start Jupyter from `training/` in a checkout, with
  `backend/requirements-ml.txt` installed and `HF_TOKEN` exported.
- **More Singlish:** the synthetic Singlish is templated. Upload jsonl files of `{"text", "label"}`
  rows (e.g. from CoSEM or the NUS SMS Corpus) and list them in `EXTRA_ARGS` as `--extra <file>`.

In [ ]:
QUICK_TRIAL = False   # True: 200 rows, 1 epoch, 100 dev rows - check the setup in a minute before the real run
EXTRA_ARGS = ""       # more style_singbert.py options, e.g. "--epochs 4" or "--extra /content/cosem.jsonl"
HUB_REPO = "makan-where-cs425/makan-singbert-style-classifier"
REPO_URL = "github.com/deniseLie/CS425-Makan-Where.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    REPO = Path("/content/CS425-Makan-Where")
    OUT = Path("/content/drive/MyDrive/makan-style/singbert" + ("_trial" if QUICK_TRIAL else ""))
    if not REPO.exists():
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{REPO_URL}"
        # no check=True: its error message would print the URL, token included
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(REPO)]).returncode:
            raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "accelerate"], check=True)
else:
    REPO = Path.cwd().parent              # run from training/ in a local checkout
    OUT = REPO / "outputs" / ("style_singbert_trial" if QUICK_TRIAL else "style_singbert")   # outputs/ is gitignored

SCRIPT = REPO / "training" / "style_singbert.py"
FINAL = OUT / "final"   # the trained model + tokenizer
assert SCRIPT.exists(), f"{SCRIPT} missing: is style_singbert.py pushed to GitHub main?"
ARGS = f"--out '{OUT}' " + ("--limit 200 --epochs 1 --eval-n 100 " if QUICK_TRIAL else "") + EXTRA_ARGS
print("script:", SCRIPT, "| out:", OUT, "| args:", ARGS)

## Train and evaluate on dev

Prints the train / dev sizes per source and label, the dev scores after each epoch, then the final dev
scores for the model and for the keyword stub (overall macro F1, and accuracy per source). Dev texts that
also occur in train are dropped first. Misclassified dev rows go to `OUT/eval_style_dev_errors.jsonl`;
the model to `OUT/final`.

In [ ]:
!python "{SCRIPT}" {ARGS}

## Look at the mistakes

In [ ]:
import json
errors = [json.loads(line) for line in open(OUT / "eval_style_dev_errors.jsonl", encoding="utf-8")]
print(len(errors), "errors")
for e in errors[:30]:
    print(f"{e['label']:8} -> {e['pred']:8} [{e['source']}] {e['text']}")

## Push to the Hub

Only once the dev scores look good. Uploads `OUT/final` (model + tokenizer, with `id2label` =
`{0: "english", 1: "singlish"}` in its config) to the private repo; the Hub keeps earlier versions in its
history. Then serve it with `MAKAN_STYLE=hf` and `MAKAN_STYLE_MODEL=makan-where-cs425/makan-singbert-style-classifier` in the
backend's `.env`.

In [ ]:
from huggingface_hub import HfApi

assert not QUICK_TRIAL, "that was a trial run: set QUICK_TRIAL = False and train for real first"
api = HfApi()
api.create_repo(HUB_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=str(FINAL), repo_id=HUB_REPO, commit_message="SingBERT style classifier")
print(f"pushed https://huggingface.co/{HUB_REPO}")